In [1]:
!pip install imbalanced-learn joblib


Defaulting to user installation because normal site-packages is not writeable


In [2]:
import pandas as pd
import numpy as np
import os
import glob
from sklearn.preprocessing import LabelEncoder, StandardScaler
from imblearn.under_sampling import RandomUnderSampler
import joblib

In [3]:
# 2. تحميل ودمج البيانات من المجلدات الفرعية
root_dir = r'D:\CSV'   # ضع مسار ملفاتك هنا
all_files = glob.glob(os.path.join(root_dir, "**", "*.csv"), recursive=True)

dfs = []
for file_path in all_files:
    try:
        folder_name = os.path.basename(os.path.dirname(file_path))
        df = pd.read_csv(file_path)
        df["label"] = folder_name
        
        # تنظيف البيانات
        df.replace([np.inf, -np.inf], np.nan, inplace=True)
        df.dropna(inplace=True)
        
        # أخذ عينة لو الملف كبير
        if len(df) > 5000:
            df = df.sample(n=5000, random_state=42)
        
        dfs.append(df)
    except:
        continue

dataset = pd.concat(dfs, ignore_index=True)
print("✅ حجم البيانات النهائي:", dataset.shape)
dataset.head()


✅ حجم البيانات النهائي: (1535578, 40)


,Header_Length,Protocol Type,Time_To_Live,Rate,fin_flag_number,syn_flag_number,rst_flag_number,psh_flag_number,ack_flag_number,ece_flag_number,...,Tot sum,Min,Max,AVG,Std,Tot size,IAT,Number,Variance,label
0,13.2,17,111.8,21.654112,0.0,0.0,0.0,0.0,0.3,0.0,...,2105.0,60.0,1392.0,210.5,415.549502,210.5,0.046181,10.0,172681.388889,Backdoor_Malware
1,11.2,17,63.5,134.621809,0.0,0.1,0.0,0.0,0.0,0.0,...,4736.0,60.0,1392.0,473.6,632.696206,473.6,0.008186,10.0,400304.488889,Backdoor_Malware
2,13.6,17,65.6,211.662495,0.0,0.1,0.0,0.0,0.2,0.0,...,3788.0,62.0,1392.0,378.8,508.763381,378.8,0.004735,10.0,258840.177778,Backdoor_Malware
3,24.8,6,84.4,155.707333,0.0,0.0,0.0,0.4,0.7,0.0,...,2917.0,62.0,833.0,291.7,308.737951,291.7,0.006422,10.0,95319.122222,Backdoor_Malware
4,10.4,17,118.3,105.440687,0.0,0.0,0.0,0.0,0.1,0.0,...,1163.0,62.0,230.0,116.3,75.052648,116.3,0.009484,10.0,5632.900000,Backdoor_Malware


In [4]:
y_raw = dataset["label"]
X_raw = dataset.drop("label", axis=1)

print("عدد الميزات:", X_raw.shape[1])
print("عدد العينات:", X_raw.shape[0])


عدد الميزات: 39
عدد العينات: 1535578


In [5]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

le = LabelEncoder()
y_encoded = le.fit_transform(y_raw)

print("✅ تم التطبيع والترميز")


✅ تم التطبيع والترميز


In [6]:
rus = RandomUnderSampler(random_state=42)
X_res, y_res = rus.fit_resample(X_scaled, y_encoded)

print("✅ بعد الموازنة:")
print("عدد العينات:", X_res.shape[0])


✅ بعد الموازنة:
عدد العينات: 42568


In [7]:
num_features = X_res.shape[1]
X_final = X_res.reshape(X_res.shape[0], num_features, 1)

print("✅ الشكل النهائي:", X_final.shape)


✅ الشكل النهائي: (42568, 39, 1)


In [9]:
save_dir = r'D:\pre prossesed data set'
if not os.path.exists(save_dir):
    os.makedirs(save_dir)

np.save(os.path.join(save_dir, 'X_cnn_ready.npy'), X_final)
np.save(os.path.join(save_dir, 'y_cnn_ready.npy'), y_res)

joblib.dump(scaler, os.path.join(save_dir, 'scaler.pkl'))
joblib.dump(le, os.path.join(save_dir, 'label_encoder.pkl'))

print("✅ تمت العملية! البيانات جاهزة للـ CNN")
print("📂 الملفات محفوظة في:", save_dir)


✅ تمت العملية! البيانات جاهزة للـ CNN
📂 الملفات محفوظة في: D:\pre prossesed data set


In [10]:
print(all_files)


['D:\\CSV\\CSV\\Backdoor_Malware\\Backdoor_Malware.pcap.csv', 'D:\\CSV\\CSV\\Benign_Final\\BenignTraffic.pcap.csv', 'D:\\CSV\\CSV\\Benign_Final\\BenignTraffic1.pcap.csv', 'D:\\CSV\\CSV\\Benign_Final\\BenignTraffic2.pcap.csv', 'D:\\CSV\\CSV\\Benign_Final\\BenignTraffic3.pcap.csv', 'D:\\CSV\\CSV\\BrowserHijacking\\BrowserHijacking.pcap.csv', 'D:\\CSV\\CSV\\CommandInjection\\CommandInjection.pcap.csv', 'D:\\CSV\\CSV\\DDoS-ACK_Fragmentation\\DDoS-ACK_Fragmentation.pcap.csv', 'D:\\CSV\\CSV\\DDoS-ACK_Fragmentation\\DDoS-ACK_Fragmentation1.pcap.csv', 'D:\\CSV\\CSV\\DDoS-ACK_Fragmentation\\DDoS-ACK_Fragmentation10.pcap.csv', 'D:\\CSV\\CSV\\DDoS-ACK_Fragmentation\\DDoS-ACK_Fragmentation11.pcap.csv', 'D:\\CSV\\CSV\\DDoS-ACK_Fragmentation\\DDoS-ACK_Fragmentation12.pcap.csv', 'D:\\CSV\\CSV\\DDoS-ACK_Fragmentation\\DDoS-ACK_Fragmentation2.pcap.csv', 'D:\\CSV\\CSV\\DDoS-ACK_Fragmentation\\DDoS-ACK_Fragmentation3.pcap.csv', 'D:\\CSV\\CSV\\DDoS-ACK_Fragmentation\\DDoS-ACK_Fragmentation4.pcap.csv', 'D:

In [ ]:
pip install tensorflow


In [12]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np


ModuleNotFoundError: No module named 'tensorflow'